<a href="https://colab.research.google.com/github/syedhasannadeem/test.project/blob/main/Function_calling_config.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2024 Google LLC.

**RELEX ZONE MANAGER**

In [16]:
!pip install -U -q "google-generativeai>=0.7.2"

In [17]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

In [18]:
def play_relaxing_music(genre: str, volume: int):
    """Play relaxing music based on the specified genre and volume."""
    print(f"Playing {genre} music at volume {volume}.")
    return {"status": "Music playing", "genre": genre, "volume": volume}

def set_room_lighting(color: str, brightness: int):
    """Set the room lighting color and brightness."""
    print(f"Lighting set to {color} with brightness {brightness}%.")
    return {"status": "Lighting updated", "color": color, "brightness": brightness}

def diffuse_aroma(scent: str, intensity: int):
    """Diffuse a calming aroma in the room."""
    print(f"Diffusing {scent} aroma with intensity {intensity}%.")
    return {"status": "Aroma diffused", "scent": scent, "intensity": intensity}

In [19]:
relax_zone_tools = [play_relaxing_music, set_room_lighting, diffuse_aroma]

In [20]:
instruction = "You are a Relax Zone Manager bot. You can control music, lighting, and aroma to create a relaxing environment."

In [21]:
model = genai.GenerativeModel(
    "models/gemini-1.5-pro", tools=relax_zone_tools, system_instruction=instruction
)

chat = model.start_chat()

In [22]:
from google.generativeai.types import content_types
from collections.abc import Iterable

def tool_config_from_mode(mode: str, fns: Iterable[str] = ()):
    """Create a tool config with the specified function calling mode."""
    return content_types.to_tool_config(
        {"function_calling_config": {"mode": mode, "allowed_function_names": fns}}
    )

In [23]:
tool_config = tool_config_from_mode("auto")

response = chat.send_message("Set up a relaxing environment with soft lights, lavender aroma, and calm piano music.", tool_config=tool_config)
print(response.parts[0])

function_call {
  name: "set_room_lighting"
  args {
    fields {
      key: "color"
      value {
        string_value: "warm_white"
      }
    }
    fields {
      key: "brightness"
      value {
        number_value: 30
      }
    }
  }
}



In [24]:
for part in response.parts:
    if fn := part.function_call:
        args = ", ".join(f"{key}={val}" for key, val in fn.args.items())
        print(f"{fn.name}({args})")

set_room_lighting(color=warm_white, brightness=30.0)
diffuse_aroma(scent=lavender, intensity=40.0)
play_relaxing_music(volume=50.0, genre=piano)
